In [1]:
import os
import re
import random
import string
import requests

import numpy as np
import pandas as pd
import geopandas as gpd

import arcpy
import arcpy.sa as sa
import arcpy.mp as mp
arcpy.CheckOutExtension("Spatial")

import time

import shutil

#### Create variables for each part of the Census API URL. This makes it easy to alter the values later.  
#### No spaces allowed!

In [2]:
#Working test URL for DHHL: https://api.census.gov/data/2022/acs/acs5?get=NAME,B01001_001E&for=american%20indian%20area/alaska%20native%20area/hawaiian%20home%20land:*

# Default values
census_base_url = "https://api.census.gov/"
census_dataset_url = "data/2022/acs/acs5"
decennial_dataset_url = "data/2020/dec/pl"
census_variables1 = "NAME,B01001_001E,B25034_006E,B25034_007E,B25034_008E,B25034_009E,B25034_010E,B25034_011E,B28002_013E,B09019_005E,B09019_008E,B19101_001E,B20005_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_020E,B01001_021E"
census_variables2 = "NAME,B01001_016E,B01001_017E,B01001_018E,B01001_019E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E"


# Geography is very important to structure correctly. Sometimes if it's incorrect, the data pull still works but
# will generate an error for higher counts of variables (like we're pulling here)

in_geography = r"&for=american%20indian%20area/alaska%20native%20area/hawaiian%20home%20land:*"

#### Combine variables to generate necessary URLs for the API pull:

In [3]:
census_combined_url = f"{census_base_url}{census_dataset_url}"
#decennial_combined_url = f"{census_base_url}{decennial_dataset_url}"

cbg_url1 = f"{census_combined_url}?get={census_variables1}{in_geography}"
cbg_url2 = f"{census_combined_url}?get={census_variables2}{in_geography}"


#### Create variables for the Census variable lookup API URLs.  
#### Variable lookup links:  
##### https://api.census.gov/data/2020/dec/pl/variables.html  
##### https://api.census.gov/data/2022/acs/acs5/variables.html  
##### https://api.census.gov/data/2022/acs/acs5/subject/variables.html    
#### Right now we just need the first 3 but may need the commented out  
#### ones if for some additional variables if we add them:

In [4]:
census_fields_url1 = f"{census_combined_url}/variables.json" #detailed variable lookup
census_fields_url2 = f"{census_combined_url}/subject/variables.json" #subject variable lookup with stats


In [5]:
print("Click link to test URL for first set of variables: " + cbg_url1 + "\r")
print("")
print("Click link to test URL for second set of variables: " + cbg_url2 + "\r")
print("")
#print("Click link to test URL for third set of variables (Decennial Census 2020): " + cbg_url3 + "\r")
#print("")

print("Click link to test URL for the field/variable list with descriptions: " + census_fields_url1 + "\r")
print("Click link to test URL for the field/variable list with descriptions: " + census_fields_url2 + "\r")
#print("Click link to test URL for the field/variable list with descriptions: " + census_fields_url3 + "\r")

Click link to test URL for first set of variables: https://api.census.gov/data/2022/acs/acs5?get=NAME,B01001_001E,B25034_006E,B25034_007E,B25034_008E,B25034_009E,B25034_010E,B25034_011E,B28002_013E,B09019_005E,B09019_008E,B19101_001E,B20005_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_020E,B01001_021E&for=american%20indian%20area/alaska%20native%20area/hawaiian%20home%20land:*

Click link to test URL for second set of variables: https://api.census.gov/data/2022/acs/acs5?get=NAME,B01001_016E,B01001_017E,B01001_018E,B01001_019E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E&for=american%20indian%20area/alaska%20native%20area/hawaiian%20home%20land:*

Click link to test URL for the field/variable list with descriptions: https://api.census.gov/data/2022/acs/acs5/variables.json
Click link to test URL for the field/variable list with descriptions: https://api.census.gov/data/2022/acs/acs5/subject/variables.json


#### Pull multiple variable listings from Census API and store each in a dataframe:

In [6]:
fields_dict1 = requests.get(census_fields_url1).json()
fields_df1 = pd.DataFrame.from_dict(fields_dict1["variables"], orient="index").reset_index()
fields_df1.head()

,index,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,B24022_060E,Estimate!!Total:!!Female:!!Service occupations...,Sex by Occupation and Median Earnings in the P...,int,B24022,0,NaN,NaN,"B24022_060EA,B24022_060M,B24022_060MA",NaN
4,B19001B_014E,"Estimate!!Total:!!$100,000 to $124,999",Household Income in the Past 12 Months (in 202...,int,B19001B,0,NaN,NaN,"B19001B_014EA,B19001B_014M,B19001B_014MA",NaN


In [7]:
fields_dict2 = requests.get(census_fields_url2).json()
fields_df2 = pd.DataFrame.from_dict(fields_dict2["variables"], orient="index").reset_index()
fields_df2.head()

,index,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,S0804_C04_068E,Estimate!!Public transportation (excluding tax...,Means of Transportation to Work by Selected Ch...,float,S0804,0,NaN,NaN,"S0804_C04_068EA,S0804_C04_068M,S0804_C04_068MA",NaN
4,S0503_C02_078E,Estimate!!Foreign born; Born in Europe!!Civili...,Selected Characteristics of the Foreign-Born P...,float,S0503,0,NaN,NaN,"S0503_C02_078EA,S0503_C02_078M,S0503_C02_078MA",NaN


#### Concatenate the multiple dataframes to create one big one with all variables:

In [8]:
fields_df = pd.concat([fields_df1, fields_df2], ignore_index=True)
print("Fields aka variables data have been concatenated")
fields_df.head()

Fields aka variables data have been concatenated


,index,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,B24022_060E,Estimate!!Total:!!Female:!!Service occupations...,Sex by Occupation and Median Earnings in the P...,int,B24022,0,NaN,NaN,"B24022_060EA,B24022_060M,B24022_060MA",NaN
4,B19001B_014E,"Estimate!!Total:!!$100,000 to $124,999",Household Income in the Past 12 Months (in 202...,int,B19001B,0,NaN,NaN,"B19001B_014EA,B19001B_014M,B19001B_014MA",NaN


### We have the field list pulled from the API but now we need the actual data  
#### Pull CBG level data from Census API and store in dataframe (first set of variables):

In [9]:
cbg_response = requests.get(cbg_url1)
cbg_data = cbg_response.json()
cbg_df = pd.DataFrame(cbg_data[1:], columns=cbg_data[0])

#Check results of API pull:
print("Status (200 is good):")
print(cbg_response.status_code)


Status (200 is good):
200


#### Rename AIANNHH field because by default it's spelled out with special characters

In [10]:
geo_col = [col for col in cbg_df.columns if 'american indian area' in col][0]

# Rename it
cbg_df.rename(columns={geo_col: 'AIANNHH'}, inplace=True)

In [11]:
cbg_df.head()

,NAME,B01001_001E,B25034_006E,B25034_007E,B25034_008E,B25034_009E,B25034_010E,B25034_011E,B28002_013E,B09019_005E,B09019_008E,B19101_001E,B20005_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_020E,B01001_021E,AIANNHH
0,"Acoma Pueblo and Off-Reservation Trust Land, NM",2982,170,158,97,34,70,265,340,29,60,583,1152,49,62,143,128,41,6,0010
1,Agua Caliente Indian Reservation and Off-Reser...,27694,7668,6846,2741,1410,153,280,1294,4276,2969,6677,15049,288,224,232,182,743,1142,0020
2,Alabama-Coushatta Reservation and Off-Reservat...,681,13,63,23,4,2,0,38,18,21,139,214,40,50,28,23,11,2,0050
3,"Allegany Reservation, NY",6135,99,239,238,279,256,1641,467,426,419,1469,2337,164,233,186,123,63,93,0080
4,"Alturas Indian Rancheria, CA",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0095


In [12]:
cbg_response2 = requests.get(cbg_url2)
cbg_data2 = cbg_response2.json()
cbg_df2 = pd.DataFrame(cbg_data2[1:], columns=cbg_data2[0])

#Check results of API pull:
print("Status (200 is good):")
print(cbg_response2.status_code)


Status (200 is good):
200


In [13]:
#### Rename AIANNHH field because by default it's spelled out with special characters

In [14]:
geo_col2 = [col for col in cbg_df2.columns if 'american indian area' in col][0]

# Rename it
cbg_df2.rename(columns={geo_col: 'AIANNHH'}, inplace=True)

cbg_df2.head()

,NAME,B01001_016E,B01001_017E,B01001_018E,B01001_019E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E,AIANNHH
0,"Acoma Pueblo and Off-Reservation Trust Land, NM",144,72,62,57,239,0,2548,0,20,27,148,0010
1,Agua Caliente Indian Reservation and Off-Reser...,1389,1699,884,1125,21659,1049,275,1283,74,1442,1912,0020
2,Alabama-Coushatta Reservation and Off-Reservat...,17,28,12,5,34,0,519,31,0,2,95,0050
3,"Allegany Reservation, NY",169,242,78,127,4117,135,1324,24,0,171,364,0080
4,"Alturas Indian Rancheria, CA",0,0,0,0,0,0,0,0,0,0,0,0095


In [15]:
cbg_df2.head()

,NAME,B01001_016E,B01001_017E,B01001_018E,B01001_019E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E,AIANNHH
0,"Acoma Pueblo and Off-Reservation Trust Land, NM",144,72,62,57,239,0,2548,0,20,27,148,0010
1,Agua Caliente Indian Reservation and Off-Reser...,1389,1699,884,1125,21659,1049,275,1283,74,1442,1912,0020
2,Alabama-Coushatta Reservation and Off-Reservat...,17,28,12,5,34,0,519,31,0,2,95,0050
3,"Allegany Reservation, NY",169,242,78,127,4117,135,1324,24,0,171,364,0080
4,"Alturas Indian Rancheria, CA",0,0,0,0,0,0,0,0,0,0,0,0095


#### Find duplicate fields in the second DF and remove all except AIANNHH which is needed to join
#### Merge tables on GEO_ID and add suffix to duplicated field (should just be GEO_ID):

In [16]:
duplicate_columns = [col for col in cbg_df2.columns if col in cbg_df.columns and col != 'AIANNHH']

# Drop them from cbg_df2
cbg_df2 = cbg_df2.drop(columns=duplicate_columns)

# Now safe to join but still add a suffix if a field is duplicated
cbg_df = cbg_df.join(cbg_df2, how='left', lsuffix='', rsuffix='_2')

#Check if it worked
cbg_df.head()

,NAME,B01001_001E,B25034_006E,B25034_007E,B25034_008E,B25034_009E,B25034_010E,B25034_011E,B28002_013E,B09019_005E,...,B01001_018E,B01001_019E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E,AIANNHH_2
0,"Acoma Pueblo and Off-Reservation Trust Land, NM",2982,170,158,97,34,70,265,340,29,...,62,57,239,0,2548,0,20,27,148,0010
1,Agua Caliente Indian Reservation and Off-Reser...,27694,7668,6846,2741,1410,153,280,1294,4276,...,884,1125,21659,1049,275,1283,74,1442,1912,0020
2,Alabama-Coushatta Reservation and Off-Reservat...,681,13,63,23,4,2,0,38,18,...,12,5,34,0,519,31,0,2,95,0050
3,"Allegany Reservation, NY",6135,99,239,238,279,256,1641,467,426,...,78,127,4117,135,1324,24,0,171,364,0080
4,"Alturas Indian Rancheria, CA",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0095


#### Move AIANNHH field to beginning of table because it's basically an index or unique ID:

In [23]:
cols = ['AIANNHH'] + [col for col in cbg_df.columns if col != 'AIANNHH']
cbg_df = cbg_df[cols]

#### Delete the extra AIANNHH_2 field. This was kept before for joining but now is no longer necessary:

In [29]:
if 'AIANNHH_2' in cbg_df.columns:
    cbg_df.drop(columns='AIANNHH_2', inplace=True)

#### Create path for output folder and store it as a variable:

In [30]:
output_folder_path = os.path.join(os.getcwd(), "output")
print ("Output folder: " + output_folder_path)

Output folder: C:\github\CCSVI\Scripts\Three_Step\output


#### Generate table names from the URL segments so that it's clear what year/data is included in the output table:

In [31]:
#general_table_name = func_clean_table_name(f"{census_dataset_url}")
#fields_table_name = f"{general_table_name}_Variables"
#cbg_table_name = f"{general_table_name}_CBG"

fields_table_name = f"census_dhhl_variables"
cbg_table_name = f"census_dhhl_data"

#### Generate output folder and use path to store CBG data pull as a CSV file

In [32]:
os.makedirs(output_folder_path, exist_ok=True)
cbg_csv_path = os.path.join(output_folder_path, f"{cbg_table_name}.csv")
cbg_df.to_csv(cbg_csv_path, index=False, encoding="utf-8")
print(f"CSV file created successfully: {cbg_csv_path}")

CSV file created successfully: C:\github\CCSVI\Scripts\Three_Step\output\census_dhhl_data.csv


#### Generate path to store Census variable list as a CSV file:

In [33]:
fields_csv_path = os.path.join(output_folder_path, f"{fields_table_name}.csv")
fields_df.to_csv(fields_csv_path, index=False, encoding="utf-8")
print(f"CSV file created successfully: {fields_csv_path}")

CSV file created successfully: C:\github\CCSVI\Scripts\Three_Step\output\census_dhhl_variables.csv


In [34]:
print("Census pull complete! Output files are located at: " + output_folder_path)

Census pull complete! Output files are located at: C:\github\CCSVI\Scripts\Three_Step\output
